In [1]:
import os
from typing import List

from dao.lab_report import DAOLabReport
from models.lab_report import LabReport, LabReportInDB

dao = DAOLabReport(collection_name="lab_reports_15-10-25")

[nltk_data] Downloading package stopwords to /home/pawel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/pawel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package pl196x to /home/pawel/nltk_data...
[nltk_data]   Package pl196x is already up-to-date!
[nltk_data] Downloading package wordnet to /home/pawel/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
def save_text_files_to_mongo(directory_path: str, is_generated: bool):
    # Check if the directory exists
    if not os.path.isdir(directory_path):
        print(f"The directory {directory_path} does not exist.")
        return
    
    # Loop through each file in the directory
    for filename in os.listdir(directory_path):
        # Check if the file has a .txt extension
        if filename.endswith(".txt"):
            file_path = os.path.join(directory_path, filename)
            # Open and read the file
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.readlines()
                non_blank_lines = [line.strip() for line in content if line.strip()]
                content = '\n'.join(non_blank_lines)
                # Save the content to MongoDB
                lab_report = LabReport(
                    plaintext_content=content,
                    tag=filename,
                    is_generated=is_generated
                )
                dao.insert_one(lab_report)

In [5]:
save_text_files_to_mongo("/mnt/d/Dev/ANANAS_data/17-11-24-real", is_generated=False)

In [ ]:
gen_lab_reports = dao.find_many_by_query({"is_generated": True})
real_lab_reports = dao.find_many_by_query({"is_generated": False})

In [ ]:
from analysis.attribute_retriving import count_em_dashes
from analysis.nlp_transformations import preprocess_text
from dao.attribute import DAOAttributePL
from models.lab_report import LabReportInDB

dao_attribute = DAOAttributePL(collection_name="attributes_new_preprocessing_30-10-25")
dao_attribute_prod = DAOAttributePL(collection_name="attributes_reference", db_name="ananas_prod")

In [ ]:
def calc_em_dashes(lab_reports: list[LabReportInDB]):
    for lab_report in lab_reports:
        preprocessed_text = preprocess_text(lab_report.plaintext_content)
        dao.update_one({"_id": lab_report.id}, {"$set": {"plaintext_content_preprocessed": preprocessed_text}})
        em_dashes, em_dashes_normalized = count_em_dashes(preprocessed_text)
        dao_attribute.update_one({"referenced_doc_id": lab_report.id},{"$set": {"number_of_em_dashes": em_dashes, "number_of_em_dashes_normalized": em_dashes_normalized} } )
        dao_attribute_prod.update_one({"referenced_doc_id": lab_report.id},{"$set": {"number_of_em_dashes": em_dashes, "number_of_em_dashes_normalized": em_dashes_normalized} } )


In [ ]:
calc_em_dashes(gen_lab_reports)

In [ ]:
calc_em_dashes(real_lab_reports)